In [16]:
!pip -q install ucimlrepo

In [17]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ucimlrepo import fetch_ucirepo

%matplotlib inline

In [18]:
np.random.seed(42)

## Scope and key decisions
- The model is a 100% NumPy MLP with manual backprop and mini-batch gradient descent updates.
- ReLU in hidden layers and Softmax in the output, per requirements.
- Loss is mini-batch mean Cross-Entropy.
- Z-score standardization uses only training statistics to avoid leakage.
- Stratified 80/20 split preserves class distribution.

## Notebook process map
1. Data Processing
2. Model Implementation
3. Training and Visualization
4. Parameter Exploration
5. Final Test
6. Interpretability

In [19]:
# Load UCI dataset
dataset = fetch_ucirepo(id=17)
X_df = dataset.data.features
y_df = dataset.data.targets

X = X_df.values.astype(np.float64)
y_raw = y_df.values.ravel()

if y_raw.dtype.kind in {"U", "S", "O"}:
    mapping = {"M": 1, "B": 0}
    y_labels = np.array([mapping.get(v, v) for v in y_raw], dtype=int)
else:
    y_labels = y_raw.astype(int)

feature_names = X_df.columns.tolist()

print("X shape:", X.shape)
print("y shape:", y_labels.shape)
print("Class counts:", np.bincount(y_labels))

X shape: (569, 30)
y shape: (569,)
Class counts: [357 212]


## Dataset and loading
- Source: UCI Breast Cancer Wisconsin (Diagnostic), via `ucimlrepo`.
- `X` has 30 numeric features; `y` is mapped to 0/1 (B=0, M=1).

In [20]:
def stratified_split(X, y, test_size=0.2, seed=42):
    """Stratified split to preserve class proportions in train/test."""
    rng = np.random.default_rng(seed)
    classes = np.unique(y)
    train_idx = []
    test_idx = []
    for cls in classes:
        cls_idx = np.where(y == cls)[0]
        rng.shuffle(cls_idx)
        n_test = int(np.ceil(test_size * len(cls_idx)))
        test_idx.append(cls_idx[:n_test])
        train_idx.append(cls_idx[n_test:])
    train_idx = np.concatenate(train_idx)
    test_idx = np.concatenate(test_idx)
    rng.shuffle(train_idx)
    rng.shuffle(test_idx)
    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]

def standardize_train_test(X_train, X_test):
    """Standardize using train mean/std to avoid leakage."""
    mean = X_train.mean(axis=0)
    std = X_train.std(axis=0)
    std[std == 0] = 1.0
    X_train_std = (X_train - mean) / std
    X_test_std = (X_test - mean) / std
    return X_train_std, X_test_std, mean, std

def one_hot(y, num_classes):
    """Convert integer labels to one-hot (num_classes)."""
    y = y.astype(int)
    out = np.zeros((y.shape[0], num_classes), dtype=np.float64)
    out[np.arange(y.shape[0]), y] = 1.0
    return out

def accuracy(y_true, y_pred):
    """Simple accuracy: fraction of correct predictions."""
    return np.mean(y_true == y_pred)

In [21]:
X_train, X_test, y_train, y_test = stratified_split(X, y_labels, test_size=0.2, seed=42)
X_train_std, X_test_std, train_mean, train_std = standardize_train_test(X_train, X_test)

num_classes = 2
y_train_oh = one_hot(y_train, num_classes)
y_test_oh = one_hot(y_test, num_classes)

print("Train shape:", X_train_std.shape, y_train_oh.shape)
print("Test shape:", X_test_std.shape, y_test_oh.shape)

Train shape: (454, 30) (454, 2)
Test shape: (115, 30) (115, 2)


## Preprocessing
- Stratified 80/20 split to keep class proportions.
- Z-score standardization using only train mean/std.
- One-hot encoding for Softmax output.

In [22]:
def relu(z):
    """Element-wise ReLU."""
    return np.maximum(0.0, z)

def relu_backward(dA, z):
    """ReLU gradient using pre-activation `z`."""
    dZ = dA.copy()
    dZ[z <= 0] = 0.0
    return dZ

def softmax(z):
    """Row-wise stable Softmax (subtract max to avoid overflow)."""
    z_shift = z - np.max(z, axis=1, keepdims=True)
    exp_z = np.exp(z_shift)
    return exp_z / np.sum(exp_z, axis=1, keepdims=True)

def cross_entropy(probs, y_true):
    """Mean Cross-Entropy per sample."""
    eps = 1e-12
    return -np.mean(np.sum(y_true * np.log(probs + eps), axis=1))

## Math engine (activations and loss)
- ReLU for hidden layers with analytic derivative for backprop.
- Numerically stable Softmax (max subtraction) to avoid overflow.
- Mini-batch mean Cross-Entropy.

In [23]:
class MLP:
    """MLP with ReLU hidden layers and Softmax output."""
    def __init__(self, layer_sizes, seed=42):
        """Initialize small weights and zero biases."""
        self.rng = np.random.default_rng(seed)
        self.W = []
        self.b = []
        for i in range(len(layer_sizes) - 1):
            in_dim = layer_sizes[i]
            out_dim = layer_sizes[i + 1]
            W = self.rng.normal(0.0, 0.01, size=(in_dim, out_dim))
            b = np.zeros((1, out_dim), dtype=np.float64)
            self.W.append(W)
            self.b.append(b)

    def forward(self, X):
        """Forward pass with caches for backward."""
        A = X
        caches = []
        for i in range(len(self.W) - 1):
            Z = A @ self.W[i] + self.b[i]
            caches.append({"A_prev": A, "Z": Z})
            A = relu(Z)
        ZL = A @ self.W[-1] + self.b[-1]
        caches.append({"A_prev": A, "Z": ZL})
        probs = softmax(ZL)
        return probs, caches

    def backward(self, probs, Y, caches):
        """Backward pass: gradients via chain rule."""
        m = Y.shape[0]
        dZ = (probs - Y) / m
        grads_W = []
        grads_b = []
        for i in reversed(range(len(self.W))):
            A_prev = caches[i]["A_prev"]
            dW = A_prev.T @ dZ
            db = np.sum(dZ, axis=0, keepdims=True)
            grads_W.insert(0, dW)
            grads_b.insert(0, db)
            if i > 0:
                dA_prev = dZ @ self.W[i].T
                Z_prev = caches[i - 1]["Z"]
                dZ = relu_backward(dA_prev, Z_prev)
        self.grads_W = grads_W
        self.grads_b = grads_b

    def update(self, lr):
        """Update parameters with plain gradient descent."""
        for i in range(len(self.W)):
            self.W[i] -= lr * self.grads_W[i]
            self.b[i] -= lr * self.grads_b[i]

    def predict_proba(self, X):
        """Return Softmax probabilities."""
        probs, _ = self.forward(X)
        return probs

    def predict(self, X):
        """Return predicted class (argmax)."""
        probs = self.predict_proba(X)
        return np.argmax(probs, axis=1)

    def fit(self, X_train, y_train_oh, y_train_labels, X_test, y_test_labels, epochs, batch_size, lr):
        """Train with mini-batches and log loss/accuracy per epoch."""
        n = X_train.shape[0]
        losses = []
        history = {"train_acc": [], "test_acc": []}
        for epoch in range(epochs):
            indices = self.rng.permutation(n)
            for start in range(0, n, batch_size):
                end = start + batch_size
                batch_idx = indices[start:end]
                Xb = X_train[batch_idx]
                yb = y_train_oh[batch_idx]
                probs, caches = self.forward(Xb)
                loss = cross_entropy(probs, yb)
                losses.append(loss)
                self.backward(probs, yb, caches)
                self.update(lr)
            train_acc = accuracy(y_train_labels, self.predict(X_train))
            test_acc = accuracy(y_test_labels, self.predict(X_test))
            history["train_acc"].append(train_acc)
            history["test_acc"].append(test_acc)
        return losses, history

    def input_gradient(self, x, class_idx):
        """Gradient of P(class_idx) wrt input for local interpretability."""
        x = x.reshape(1, -1)
        probs, caches = self.forward(x)
        p = probs[0]
        e = np.zeros_like(p)
        e[class_idx] = 1.0
        dZ = (p[class_idx] * (e - p)).reshape(1, -1)
        dA_prev = dZ @ self.W[-1].T
        for i in reversed(range(len(self.W) - 1)):
            Z = caches[i]["Z"]
            dZ = relu_backward(dA_prev, Z)
            dA_prev = dZ @ self.W[i].T
        return dA_prev.reshape(-1), probs